In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [2]:
import torch

# 2. Check whether a GPU is available and identify its type and memory.
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"=== GPU Status ===")
    print(f"GPU Available: {gpu_name}")
    print(f"Total VRAM: {gpu_memory:.2f} GB")
else:
    print("No GPU detected! Please ensure your Colab runtime is set to GPU (T4, L4, or A100).")

=== GPU Status ===
GPU Available: Tesla T4
Total VRAM: 14.56 GB


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 3. Choose a small, instruction-tuned causal language model
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading tokenizer for Qwen/Qwen2.5-0.5B-Instruct...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Error: No runtime connected'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
from datasets import Dataset

# 4. Create a small dataset of at least 30 question-and-answer pairs about a specific domain (Computer Shop)
raw_data = {
    "question": [
        "What are your store opening hours?",
        "Do you build custom gaming PCs?",
        "What is the warranty policy on graphics cards?",
        "Do you offer laptop repair services?",
        "Where is your shop located?",
        "Do you sell mechanical keyboards?",
        "Can I pay using credit cards or installment plans?",
        "Do you sell M.2 NVMe SSDs?",
        "What graphics cards do you currently have in stock?",
        "Do you buy used laptops or PC parts?",
        "How long does a standard PC diagnostic take?",
        "Do you sell pre-built office desktops?",
        "Are your PC components brand new with manufacturer warranty?",
        "Do you provide data recovery services from damaged hard drives?",
        "Can I bring my own parts for you to assemble?",
        "Do you sell DDR5 RAM modules?",
        "What is your return policy for unopened items?",
        "Do you sell liquid cooling kits for CPUs?",
        "Do you offer on-site IT support for businesses?",
        "Do you stock wireless gaming mice?",
        "Can I reserve an item online and pick it up in-store?",
        "Do you sell 4K computer monitors?",
        "What power supply units do you recommend for high-end gaming?",
        "Do you clean dusty laptops and apply new thermal paste?",
        "Do you sell external hard drives and backup storage?",
        "Can you upgrade my laptop's RAM and storage?",
        "Do you stock budget-friendly graphics cards for casual gaming?",
        "Do you sell enterprise networking routers and switches?",
        "What brand of motherboards do you mostly carry?",
        "Do you offer student discounts on accessories?"
    ],
    "answer": [
        "We are open Monday to Saturday from 9:00 AM to 7:00 PM.",
        "Yes, we build custom high-performance gaming PCs tailored to your budget and specifications.",
        "Graphics cards come with a standard 3-year manufacturer warranty, plus a 1-year store support warranty.",
        "Yes, we handle hardware repairs, screen replacements, and software troubleshooting for all major laptop brands.",
        "Our shop is located centrally in Homagama, Sri Lanka.",
        "Yes, we stock a wide variety of mechanical keyboards featuring blue, brown, and red switches.",
        "Yes, we accept major credit cards and offer flexible monthly installment plans with select local banks.",
        "Yes, we carry fast M.2 NVMe PCIe 4.0 SSDs ranging from 500GB to 2TB capacities.",
        "We stock various NVIDIA RTX and AMD Radeon series cards. Please check our counter or website for live inventory.",
        "Yes, we evaluate and buy working used laptops and selected PC components after proper testing.",
        "Standard PC diagnostics typically take between 24 to 48 hours.",
        "Yes, we offer reliable pre-built office desktops pre-installed with operating systems.",
        "Yes, all our components are 100% brand new and backed by official manufacturer warranties.",
        "Yes, we offer professional data recovery services for corrupted or failing storage drives.",
        "Yes! You can bring your own parts, and our technicians will assemble and stress-test the PC for a nominal fee.",
        "Yes, we carry high-speed DDR5 RAM modules in 16GB and 32GB kits.",
        "Unopened items can be returned within 7 days of purchase with the original receipt for a full refund or exchange.",
        "Yes, we stock both All-in-One (AIO) liquid coolers and custom loop components.",
        "Yes, we provide corporate IT support, networking setups, and maintenance contracts for local businesses.",
        "Yes, we have a variety of ergonomic and wireless gaming mice from top brands like Logitech and Razer.",
        "Yes, you can drop us a message or call us to reserve items for in-store pickup.",
        "Yes, we sell 27-inch and 32-inch 4K IPS monitors ideal for content creation and gaming.",
        "We generally recommend fully modular PSUs rated at 650W to 850W 80+ Gold for high-end gaming builds.",
        "Yes, our laptop maintenance service includes thorough dust cleaning and high-performance thermal paste replacement.",
        "Yes, we stock portable external HDDs and high-speed SSDs from 1TB to 5TB.",
        "Yes, we can upgrade your laptop's RAM and swap out your hard drive for a faster SSD while you wait or within a day.",
        "Yes, we stock budget-friendly cards like the GTX 1650, RTX 3050, and RX 6600.",
        "Yes, we carry enterprise-grade Wi-Fi routers, switches, and patch cables.",
        "We primarily carry ASUS, MSI, Gigabyte, and ASRock motherboards.",
        "Yes, we offer a 5% discount on peripherals and accessories for students with a valid university ID."
    ]
}

dataset = Dataset.from_dict(raw_data)

# Format dataset for Instruction tuning
def format_prompt(sample):
    text = f"### Instruction:\n{sample['question']}\n\n### Response:\n{sample['answer']}"
    return {"text": text}

formatted_dataset = dataset.map(format_prompt)
print(f"Dataset successfully created with {len(dataset)} examples.")

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Dataset successfully created with 30 examples.


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

# 1. Load config and force float16 to override Qwen's default bfloat16 config
config = AutoConfig.from_pretrained(model_id)
config.torch_dtype = torch.float16

# 2. Configure QLoRA with explicit Float16 compute dtype
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load model using 'dtype' (fixes the deprecation warning) and the forced config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

# 4. PEFT LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 5. Training Arguments: Disable fp16 and bf16 scalers entirely to bypass the GradScaler bug
training_args = SFTConfig(
    output_dir="./lora_computer_shop",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=3,
    bf16=False,
    fp16=False,  # Disables GradScaler completely so it never crashes on bfloat16 tensors
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
)

# 6. Initialize SFTTrainer with SFTConfig
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting Fine-Tuning...")
trainer.train()
print("Fine-Tuning Complete!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Starting Fine-Tuning...


Step,Training Loss
5,2.832492
10,1.644678


Fine-Tuning Complete!


In [11]:
import torch

# Test questions (including seen and unseen questions)
test_questions = [
    "What are your store opening hours?",                 # Seen question
    "Do you repair liquid-damaged mechanical keyboards?", # Unseen question
    "Do you sell custom water cooling loops?"             # Unseen question
]

print("=== EVALUATION OF FINE-TUNED MODEL ===")
model.eval()

for q in test_questions:
    prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n[Question]: {q}\n{response}\n" + "-"*50)

=== EVALUATION OF FINE-TUNED MODEL ===

[Question]: What are your store opening hours?
### Instruction:
What are your store opening hours?

### Response:
We operate 7 days a week, from Monday to Saturday, with no extended hours.
--------------------------------------------------

[Question]: Do you repair liquid-damaged mechanical keyboards?
### Instruction:
Do you repair liquid-damaged mechanical keyboards?

### Response:
Yes, we can replace your keyboard if it's damaged.
--------------------------------------------------

[Question]: Do you sell custom water cooling loops?
### Instruction:
Do you sell custom water cooling loops?

### Response:
Yes, we offer high-quality, professional-grade water coolers designed for your specific needs.
--------------------------------------------------
